# Local RAG wiki with hybrid search

## Prerequisites

1. Edit `.env` in this folder. It starts with local defaults. To use Yandex Cloud, edit the same file: set `RAG_WIKI_MODEL_PROVIDER=yc`, the cloud YDB endpoint/database and YDB credentials, plus `YC_FOLDER_ID` and `YC_API_KEY` for YandexGPT. The YC API key does not authenticate YDB.
2. For local mode, start [Ollama](https://ollama.com/download), pull the model named by `RAG_WIKI_LLM_MODEL`, and start a YDB version with [HybridRank](https://ydb.tech/docs/en/dev/hybrid-search?version=main), for example with the [Docker Compose file](../../docker-compose.yml).
3. Run this notebook from `examples/local_rag_wiki_hybrid`. It installs the current repository checkout.

By default the notebook recreates the table named by `YDB_TABLE` on each run; set `YDB_DROP_EXISTING_TABLE=false` to reuse it. The `.env` sets `RAG_WIKI_DOCUMENTS=20` for a short local run; increase it if you want the full 10,000-document example. The saved outputs used 20 documents and `qwen2.5:3b` because `llama3.1` was not installed in the verification environment.

In [1]:
%pip install -q -e ../.. "langchain>=0.3,<0.4" "langchain-huggingface>=0.1,<0.2" "langchain-ollama>=0.2,<0.3" "langchain-community>=0.3,<0.4" datasets sentence-transformers yandexcloud python-dotenv

import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env", override=False)
os.environ["GRPC_VERBOSITY"] = "NONE"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## Prepare dataset

Real dataset from [huggingface](https://huggingface.co/datasets/Cohere/wikipedia-22-12-simple-embeddings/viewer/default/train?p=2&views%5B%5D=train): 

In [2]:
from datasets import load_dataset

ds = load_dataset("Cohere/wikipedia-22-12-simple-embeddings")
ds["train"][0]["text"]

'The 24-hour clock is a way of telling the time in which the day runs from midnight to midnight and is divided into 24 hours, numbered from 0 to 23. It does not use a.m. or p.m. This system is also referred to (only in the US and the English speaking parts of Canada) as military time or (only in the United Kingdom and now very rarely) as continental time. In some parts of the world, it is called railway time. Also, the international standard notation of time (ISO 8601) is based on this format.'

To simplify local example, we will use only a subset from this dataset

In [3]:
from langchain_core.documents import Document

N = int(os.environ.get("RAG_WIKI_DOCUMENTS", "10000"))
documents_to_upload = [Document(ds["train"][i]["text"]) for i in range(N)]

documents_to_upload[123 if N > 123 else 0]

Document(metadata={}, page_content='The 24-hour clock is a way of telling the time in which the day runs from midnight to midnight and is divided into 24 hours, numbered from 0 to 23. It does not use a.m. or p.m. This system is also referred to (only in the US and the English speaking parts of Canada) as military time or (only in the United Kingdom and now very rarely) as continental time. In some parts of the world, it is called railway time. Also, the international standard notation of time (ISO 8601) is based on this format.')

Fake dataset from local file

In [4]:
with open("fake_wiki_ydb.md") as file:
    fake_data = file.read()

fake_data[:500]

'## Overview\nYDB is a fictional technology campus city designed as a dedicated space for innovation, work, and everyday life. YDB is located in a neutral zone and operates outside the jurisdiction of any nation-state. Officially, YDB presents itself as an independent innovation territory.\n\n## Name origin\nThe name YDB is not officially decoded. In project documentation, YDB appears as a placeholder code from an international design competition. There are rumors that YDB refers to an internal joke '

In [5]:
from langchain.text_splitter import MarkdownHeaderTextSplitter

markdown_header_text_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("##", "Header")],
)

markdown_document_splits = markdown_header_text_splitter.split_text(
    fake_data
)

markdown_document_splits[0]

Document(metadata={'Header': 'Overview'}, page_content='YDB is a fictional technology campus city designed as a dedicated space for innovation, work, and everyday life. YDB is located in a neutral zone and operates outside the jurisdiction of any nation-state. Officially, YDB presents itself as an independent innovation territory.')

Let's merge real and fake wiki

In [6]:
documents_to_upload.extend(markdown_document_splits)

## Prepare vector store

Choose the embedding model from `RAG_WIKI_MODEL_PROVIDER`. Local mode uses a cached Hugging Face model; cloud mode uses YandexGPT document and query embeddings with `YC_FOLDER_ID` and `YC_API_KEY`. Cloud embedding calls pause 0.25 seconds by default to stay below request-rate quotas; set `YC_EMBEDDING_REQUEST_INTERVAL` in `.env` to change this.

In [7]:
provider = os.getenv("RAG_WIKI_MODEL_PROVIDER", "local").lower()
if provider == "local":
    from langchain_huggingface import HuggingFaceEmbeddings

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-mpnet-base-v2"
    )
elif provider == "yc":
    if not os.getenv("YC_FOLDER_ID") or not os.getenv("YC_API_KEY"):
        raise ValueError("YC_FOLDER_ID and YC_API_KEY are required for cloud models")
    from langchain_community.embeddings.yandex import YandexGPTEmbeddings

    embeddings = YandexGPTEmbeddings(
        sleep_interval=float(os.getenv("YC_EMBEDDING_REQUEST_INTERVAL", "0.25"))
    )
else:
    raise ValueError("RAG_WIKI_MODEL_PROVIDER must be 'local' or 'yc'")

Create a YDB vector store with hybrid search enabled. The store creates a `fulltext_relevance` index on the document text and a `vector_kmeans_tree` index on the embeddings. The vector dimension is inferred from the selected embedding model.

In [8]:
import ydb.iam
from langchain_ydb.vectorstores import YDB, YDBSettings


def env_bool(name, default="false"):
    return os.getenv(name, default).lower() in {"1", "true", "yes"}


iam_token = os.getenv("YDB_IAM_TOKEN")
service_account_file = os.getenv("YDB_SERVICE_ACCOUNT_KEY_FILE")
if iam_token and service_account_file:
    raise ValueError("Set either YDB_IAM_TOKEN or YDB_SERVICE_ACCOUNT_KEY_FILE")
if service_account_file:
    ydb_credentials = ydb.iam.ServiceAccountCredentials.from_file(service_account_file)
elif iam_token:
    ydb_credentials = {"token": iam_token}
else:
    ydb_credentials = None

vector_store = YDB(
    embeddings,
    config=YDBSettings(
        host=os.getenv("YDB_HOST", "localhost"),
        port=int(os.getenv("YDB_PORT", "2136")),
        database=os.getenv("YDB_DATABASE", "/local"),
        table=os.getenv("YDB_TABLE", "langchain_ydb_local_rag_hybrid"),
        secure=env_bool("YDB_SECURE"),
        credentials=ydb_credentials,
        hybrid_search_enabled=True,
        index_config_clusters=64,
        index_tree_search_top_size=10,
        drop_existing_table=env_bool("YDB_DROP_EXISTING_TABLE", "true"),
    ),
)

Finally, let's load prepared documents:

In [9]:
ids = vector_store.add_documents(documents_to_upload, batch_size=100)

Let's check how this vector store works:

In [10]:
vector_store.similarity_search("social network", k=1)

[Document(id='fdfc864f35b1e06ca155b3b3fa85b54f3b8a83c0', metadata={'Header': 'Cultural life'}, page_content='Despite its tech focus, YDB supports an active cultural scene. YDB regularly hosts festivals, digital art shows, public readings, and live audio-visual performances. Culture is physically embedded in YDB’s architecture, coexisting with research and workspaces.')]

In [11]:
vector_store.similarity_search("YDB city", k=1)

[Document(id='de3264542b7551ad439fb396874cd389630a6323', metadata={'Header': 'Overview'}, page_content='YDB is a fictional technology campus city designed as a dedicated space for innovation, work, and everyday life. YDB is located in a neutral zone and operates outside the jurisdiction of any nation-state. Officially, YDB presents itself as an independent innovation territory.')]

## Compare vector and hybrid search

The two searches use the same text query. Vector search ranks by embedding similarity; YDB's `HybridRank` also retrieves and ranks text matches from the fulltext index. `weights=(2.0, 1.0)` gives the text branch twice the weight of the vector branch.

In [12]:
query = "licensed kiosks"
vector_results = vector_store.similarity_search(query, k=3)
hybrid_results = vector_store.hybrid_search(query, k=3, weights=(2.0, 1.0))

{
    "vector": [(doc.metadata.get("Header", "Wikipedia"), doc.page_content[:120]) for doc in vector_results],
    "hybrid": [(doc.metadata.get("Header", "Wikipedia"), doc.page_content[:120]) for doc in hybrid_results],
}

{'vector': [('Food systems',
   'Food services in YDB are operated by automated kitchens, smart cafeterias, and vertical farms. Meals in YDB are personal'),
  ('Technology ecosystem',
   'All systems in YDB run on city-owned digital platforms. Smart logistics, adaptive lighting, environmental controls, and '),
  ('Transportation',
   'Public transport in YDB includes autonomous shuttles, delivery drones, and underground capsules. Private cars are not pe')],
 'hybrid': [('Currency',
   'YDB uses a native digital currency for all transactions within the city. This internal currency in YDB is pegged to syst'),
  ('Food systems',
   'Food services in YDB are operated by automated kitchens, smart cafeterias, and vertical farms. Meals in YDB are personal'),
  ('Technology ecosystem',
   'All systems in YDB run on city-owned digital platforms. Smart logistics, adaptive lighting, environmental controls, and ')]}

The ordinary `as_retriever()` uses vector search. `as_hybrid_retriever()` runs the combined fulltext and vector query. Both return LangChain `Document` objects and can be used in the RAG chain below.

In [13]:
vector_retriever = vector_store.as_retriever(search_kwargs={"k": 4})
hybrid_retriever = vector_store.as_hybrid_retriever(k=4, weights=(2.0, 1.0))

{
    "vector": [doc.metadata.get("Header", "Wikipedia") for doc in vector_retriever.invoke("licensed kiosks")],
    "hybrid": [doc.metadata.get("Header", "Wikipedia") for doc in hybrid_retriever.invoke("licensed kiosks")],
}

{'vector': ['Food systems',
  'Technology ecosystem',
  'Transportation',
  'Cultural life'],
 'hybrid': ['Currency',
  'Food systems',
  'Technology ecosystem',
  'Transportation']}

## Prepare LLM

Use Ollama for local generation or YandexGPT for cloud generation. `YC_API_KEY` is for the model API; YDB uses its separate IAM token or service account key.

In [14]:
if provider == "local":
    from langchain_ollama.llms import OllamaLLM

    llm = OllamaLLM(model=os.getenv("RAG_WIKI_LLM_MODEL", "llama3.1"))
else:
    from langchain_community.llms import YandexGPT

    llm = YandexGPT()

llm.invoke("what is YDB city?")

'I apologize for any confusion, but there seems to be an error in your query as I couldn\'t find information on "YDB city" related to Alibaba Cloud or any other prominent tech companies.\n\nIt\'s possible you may be thinking of something specific like a city with the name "YBD," which doesn\'t correspond to any known cities. If this is the case, please provide more details and I\'ll do my best to assist further.\n\nAlternatively, if you meant something else related to YDB (which likely refers to the Yet Another Database), it could be confused for another product or service by mistake. YDB stands for Yet Another Database and is a database engine designed for handling large-scale data processing in distributed environments. If you have more context or are thinking of a specific feature or application, please provide additional details so I can assist you better.'

## All Together: RAG chain

In [15]:
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser
from langchain.prompts import ChatPromptTemplate

system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentence maximum and keep the answer concise. "
    "Context: {context}"
)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def make_chain(retriever):
    return (
        {
            "context": retriever | format_docs,
            "input": RunnablePassthrough(),
        }
        | prompt
        | llm
        | StrOutputParser()
    )

vector_chain = make_chain(vector_retriever)
hybrid_chain = make_chain(hybrid_retriever)
hybrid_chain.invoke("What is YDB city?")

"YDB is a fictional technology campus city designed as an innovation space outside any nation-state's jurisdiction. It uses modular design with interconnected clusters blending residential, work, and research functions. YDB prioritizes pedestrian movement and digital integration, managed through a decentralized governance model using smart contracts and internal voting platforms."

In [16]:
question = "Where can I exchange YDB credits for fiat currency?"
{
    "vector": vector_chain.invoke(question),
    "hybrid": hybrid_chain.invoke(question),
}

{'vector': 'You can exchange YDB credits for fiat currencies at licensed kiosks.',
 'hybrid': 'You can exchange YDB credits for fiat currencies at licensed kiosks.'}

In [17]:
hybrid_chain.invoke("Where can I exchange money in YDB city?")

'You can exchange YDB credits for fiat currencies at licensed kiosks in YDB city.'